# Lab 9: Automate Weather Data Retrieval with API

### 🌤️ What is an API?
Think of an **API** (Application Programming Interface) as a waiter in a restaurant. You (the customer) tell the waiter what you want from the menu. The waiter takes your request to the kitchen (the server), and then brings the food back to you. In our case, the API is the waiter fetching live weather data from a huge database on the internet and bringing it back to our Python code!

### 🌡️ What do Weather APIs do?
Weather APIs like **OpenWeatherMap** collect data from satellites, weather stations, and sensors all over the world. They let us ask questions like "How hot is it in London?" or "Is it raining in Tokyo?" and give us back precise numbers and descriptions.

### 🔑 Why do we need API Keys?
An **API Key** is like a digital library card or a password. It tells the service provider (OpenWeatherMap) who is asking for the data. This helps them prevent abuse and ensure their servers aren't overloaded.

### 🚀 What we will build & learn
In this lab, we are building a **Weather Data Retrieval System**.
**By the end, you will know how to:**
1. Connect Python to the internet to talk to other services.
2. Handle **JSON** (the language of data on the web).
3. Create professional-looking tables in your output.
4. Save your results to a text file automatically.

## Objectives
- Understand the fundamentals of REST APIs and HTTP requests.
- Register for and obtain API keys from OpenWeatherMap.
- Make HTTPS requests using the `requests` library.
- Parse JSON responses and extract specific weather data.
- Handle errors and format data into readable tables.

## Prerequisites
- Basic Python knowledge (variables, loops, dictionaries).
- Internet connection.
- A free account at [OpenWeatherMap](https://openweathermap.org/api).

## Environment Setup
We need to install the `tabulate` library to make our weather data look pretty in tables. The `requests` library is usually pre-installed in Colab, but we will verify it.

In [6]:
# Install the tabulate library for pretty table formatting
!pip install tabulate

import requests
import json
from tabulate import tabulate
import sys
from datetime import datetime
import re
import os

# Verification of installation
print("--- Verification ---")
print(f"Requests version: {requests.__version__}")
print("Tabulate imported successfully!")

--- Verification ---
Requests version: 2.32.4
Tabulate imported successfully!


## Task 1: API Configuration
Before running the app, you need an API key.
1. Go to [OpenWeatherMap](https://home.openweathermap.org/api_keys).
2. Copy your key.
3. Paste it into the `API_KEY` variable below.

In [11]:
# @title API Configuration { vertical-output: true }
# Enter your OpenWeatherMap API key in the text box on the right
API_KEY = "" # @param {type:"string"}
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

if not API_KEY or API_KEY == "":
    print("⚠️ WARNING: You haven't entered your API key in the text box yet!")
else:
    print("✅ API Key is set and ready to use.")

⚠️ WARNING: You haven't entered your API key in the text box yet!


## Task 2 - 5: Building the Core Logic
We will now define several functions that handle validation, data fetching, extraction, and formatting. This modular approach makes our code organized and easy to debug.

In [8]:
def validate_city_name(city_name):
    """
    Validates city names using Regular Expressions.
    Allows names like 'London', 'St. Louis', or 'London,UK'.
    """
    if not city_name or not city_name.strip():
        return False
    # This pattern allows letters, spaces, commas, hyphens, and dots
    pattern = r"^[a-zA-Z\s,\-\\.']+"
    return bool(re.match(pattern, city_name.strip()))

def get_weather_data(city_name):
    """Fetches raw JSON weather data from the API."""
    if not validate_city_name(city_name):
        print(f"❌ Invalid city name format: {city_name}")
        return None

    try:
        # We use units=metric to get Celsius
        url = f"{BASE_URL}?q={city_name}&appid={API_KEY}&units=metric"
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            return response.json()
        elif response.status_code == 404:
            print(f"❌ City not found: {city_name}")
        elif response.status_code == 401:
            print("❌ Unauthorized: Check your API Key.")
        else:
            print(f"❌ API Error {response.status_code} for {city_name}")
        return None
    except Exception as e:
        print(f"❌ Network/Connection error: {e}")
        return None

def extract_weather_info(weather_data):
    """Parses the raw JSON into a clean Python dictionary."""
    if not weather_data:
        return None
    try:
        return {
            'City': weather_data['name'],
            'Country': weather_data['sys']['country'],
            'Temp (°C)': weather_data['main']['temp'],
            'Humidity (%)': weather_data['main']['humidity'],
            'Weather': weather_data['weather'][0]['description'].title(),
            'Wind (m/s)': weather_data['wind']['speed'],
            'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
    except KeyError as e:
        print(f"❌ Error parsing data: Missing field {e}")
        return None

## Task 6 - 9: Display and Export Logic
These functions calculate summary statistics (like average temperature) and save our findings to a text file.

In [9]:
def display_results(results):
    """Prints the data in a table and shows statistics."""
    if not results:
        print("No data to display.")
        return

    # Display Table
    print("\n" + "="*40)
    print("LATEST WEATHER REPORT")
    print("="*40)
    print(tabulate(results, headers="keys", tablefmt="grid"))

    # Calculate Summary Stats
    temps = [r['Temp (°C)'] for r in results]
    avg_temp = sum(temps) / len(temps)
    print(f"\nSummary: Processed {len(results)} cities.")
    print(f"Average Temperature: {avg_temp:.2f}°C")

def export_to_file(results, filename="weather_data.txt"):
    """Saves the results to a file in the Colab environment."""
    try:
        with open(filename, 'w') as f:
            f.write(f"Weather Export - {datetime.now()}\n" + "-"*30 + "\n")
            for r in results:
                f.write(f"{r['City']}, {r['Country']}: {r['Temp (°C)']}°C, {r['Weather']}\n")
        print(f"✅ Successfully exported data to {filename}")
    except Exception as e:
        print(f"❌ Export failed: {e}")

## Task 10: Run the Enhanced Weather Application
This cell combines everything. You can either enter your own cities or press Enter to use the default list.

In [10]:
# @title Run Weather App
user_input = input("Enter cities separated by commas (or press Enter for defaults): ").strip()

if not user_input:
    cities = ['London', 'New York', 'Tokyo', 'Sydney', 'Paris', 'St. Louis']
    print(f"Using default cities: {cities}")
else:
    cities = [c.strip() for c in user_input.split(',')]

final_results = []
for city in cities:
    print(f"Fetching {city}...")
    raw = get_weather_data(city)
    clean = extract_weather_info(raw)
    if clean:
        final_results.append(clean)

if final_results:
    display_results(final_results)
    export_to_file(final_results)
else:
    print("No valid weather data was retrieved. Check your API key and city names.")

Enter cities separated by commas (or press Enter for defaults): Bengaluru, Aligarh, Makkah, Jeddah, Jubail
Fetching Bengaluru...
Fetching Aligarh...
Fetching Makkah...
❌ City not found: Makkah
Fetching Jeddah...
Fetching Jubail...

LATEST WEATHER REPORT
+-----------+-----------+-------------+----------------+-----------------+--------------+---------------------+
| City      | Country   |   Temp (°C) |   Humidity (%) | Weather         |   Wind (m/s) | Timestamp           |
+===========+===========+=============+================+=================+==============+=====================+
| Bengaluru | IN        |       25.77 |             61 | Few Clouds      |         4.02 | 2026-04-13 19:33:53 |
+-----------+-----------+-------------+----------------+-----------------+--------------+---------------------+
| Aligarh   | IN        |       26.3  |             19 | Overcast Clouds |         3.12 | 2026-04-13 19:33:53 |
+-----------+-----------+-------------+----------------+-----------------+

## Verification Checklist
- [ ] `requests` and `tabulate` imported without error?
- [ ] API Key variable updated?
- [ ] Did the table display at least one city's data?
- [ ] Check the file sidebar (📁 icon on the left). Does `weather_data.txt` exist?

## Troubleshooting
- **401 Error:** Your API key is likely wrong or not yet activated (it can take up to 2 hours for new keys).
- **404 Error:** The city name is misspelled.
- **Network Timeout:** Colab might be having trouble reaching the internet. Try running the cell again.
- **NameError:** Make sure you ran all the "def" cells above before running the Main App.

## Conclusion
Congratulations! You've built a real-world tool that interacts with live data.

### Key Takeaways:
1. **APIs are powerful:** They let your code leverage data from multi-billion dollar companies.
2. **Validation is vital:** Users will type things wrong; your code must handle it gracefully.
3. **Formatting matters:** Raw JSON is for computers; tables and summaries are for humans.

### What I Learned in This Lab
*From the learner's perspective:*
I learned how to use Python's `requests` library to fetch data from a URL using HTTPS. I mastered the art of parsing complex JSON dictionaries to find specific values like temperature and wind speed. Most importantly, I learned how to keep my API keys safe and how to handle errors so my program doesn't crash when it encounters an invalid city name.